# 🛡️ CyberGuard ML URL Scanner — Model Evaluation & Comparison

**Author:** Harpreet Singh

This notebook goes deeper than the main CyberGuard notebook. It:
1. Explores the data (EDA)
2. Builds URL features by hand
3. Compares multiple ML models
4. Evaluates them (precision, recall, ROC, confusion matrix)
5. Analyses the mistakes
6. Shows which features matter most

> **Dataset:** set `DATA_PATH` below to your CSV with columns `url` and `label` (`1` = malicious, `0` = safe).
> If the file is not found, a small **synthetic demo dataset** is generated so the notebook still runs end to end.

In [ ]:
import re, math, warnings
from urllib.parse import urlparse, parse_qsl
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("Setup done. XGBoost available:", HAS_XGB)

## 1. Load data

In [ ]:
import os, random

DATA_PATH = "urls.csv"   # <-- change this to your dataset (columns: url, label)

def make_demo_data(n=1500, seed=RANDOM_STATE):
    rnd = random.Random(seed)
    good_domains = ["google.com", "wikipedia.org", "github.com", "python.org", "nptel.ac.in",
                    "amazon.in", "youtube.com", "stackoverflow.com", "microsoft.com", "kali.org"]
    good_paths = ["", "about", "docs/index", "search?q=python", "courses/ml", "blog/2024/post"]
    bad_words = ["login", "verify", "secure", "update", "account", "bank", "free", "bonus"]
    rows = []
    for _ in range(n):
        if rnd.random() < 0.5:
            url = f"https://www.{rnd.choice(good_domains)}/{rnd.choice(good_paths)}"
            rows.append((url, 0))
        else:
            host = rnd.choice([
                ".".join(str(rnd.randint(1, 254)) for _ in range(4)),
                "-".join(rnd.sample(bad_words, 2)) + rnd.choice([".xyz", ".top", ".info", ".ru"]),
                f"{rnd.choice(bad_words)}.{rnd.choice(good_domains).split('.')[0]}-{rnd.randint(10,999)}.com",
            ])
            scheme = rnd.choice(["http", "http", "https"])
            path = "/".join(rnd.sample(bad_words, rnd.randint(1, 3))) + f"?id={rnd.randint(1000,99999)}&ref=@{rnd.randint(1,99)}"
            rows.append((f"{scheme}://{host}/{path}", 1))
    return pd.DataFrame(rows, columns=["url", "label"])

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)[["url", "label"]].dropna().drop_duplicates()
    print(f"Loaded real dataset: {DATA_PATH}")
else:
    df = make_demo_data()
    print("⚠️ Dataset not found — using synthetic DEMO data (results are not realistic).")

df["url"] = df["url"].astype(str)
print(df.shape)
df.head()

## 2. Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df["label"].value_counts().rename({0: "Safe", 1: "Malicious"}).plot(kind="bar", ax=ax[0], color=["#4c9", "#e55"])
ax[0].set_title("Class balance"); ax[0].set_ylabel("Count"); ax[0].tick_params(axis="x", rotation=0)

df["len"] = df["url"].str.len()
sns.histplot(data=df, x="len", hue="label", bins=40, ax=ax[1], palette=["#4c9", "#e55"])
ax[1].set_title("URL length distribution")
plt.tight_layout(); plt.show()

## 3. Feature engineering
We turn each URL into numbers the model can learn from.

In [ ]:
SUSPICIOUS = ["login", "verify", "secure", "update", "account", "bank", "free", "bonus",
              "signin", "confirm", "password", "paypal", "wallet", "invoice", "gift"]
BRANDS = ["paypal", "google", "amazon", "microsoft", "apple", "facebook", "netflix",
          "sbi", "hdfc", "icici", "whatsapp", "instagram"]
SHORTENERS = {"bit.ly", "tinyurl.com", "goo.gl", "t.co", "ow.ly", "is.gd", "cutt.ly", "rb.gy"}
RISKY_TLDS = {"xyz", "top", "info", "ru", "tk", "ml", "ga", "cf", "gq", "click", "work", "zip", "buzz"}
IP_RE = re.compile(r"^\d{1,3}(\.\d{1,3}){3}$")

def entropy(s):
    if not s: return 0.0
    counts = Counter(s)
    return -sum((c / len(s)) * math.log2(c / len(s)) for c in counts.values())

def extract_features(url):
    p = urlparse(url if "://" in url else "http://" + url)
    host = (p.hostname or "").lower()
    path, query = p.path.lower(), p.query.lower()
    low = url.lower()
    parts = host.split(".")
    tld = parts[-1] if len(parts) > 1 else ""
    main_domain = parts[-2] if len(parts) > 1 else host
    subdomain = ".".join(parts[:-2])
    try:
        port = p.port
    except ValueError:
        port = None
    tokens = [t for t in re.split(r"[\W_]+", low) if t]
    vowels = sum(ch in "aeiou" for ch in main_domain)
    query_pairs = parse_qsl(p.query, keep_blank_values=True)
    query_keys = [k.lower() for k, _ in query_pairs]
    path_segments = [seg for seg in path.split("/") if seg]
    # A brand appearing in the hostname is not proof of legitimacy; this is only a model signal.
    hostname_without_www = host[4:] if host.startswith("www.") else host
    domain_label = parts[-2] if len(parts) > 1 else host
    brand_domain_mismatch = int(any(b in hostname_without_www for b in BRANDS)
                                 and not any(b in domain_label for b in BRANDS))
    return {
        # --- length features ---
        "url_length": len(url),
        "host_length": len(host),
        "path_length": len(path),
        "query_length": len(query),
        # --- character counts ---
        "num_dots": url.count("."),
        "num_hyphens": url.count("-"),
        "num_at": url.count("@"),
        "num_digits": sum(ch.isdigit() for ch in url),
        "num_uppercase": sum(ch.isupper() for ch in url),
        "num_special": len(re.findall(r"[^A-Za-z0-9]", url)),
        "num_slashes": url.count("/"),
        "num_equals": url.count("="),
        "digit_ratio": sum(ch.isdigit() for ch in url) / max(len(url), 1),
        # --- structure ---
        "num_params": query.count("&") + (1 if query else 0),
        "num_subdomains": max(len(parts) - 2, 0),
        "path_depth": path.count("/"),
        "double_slash_in_path": int("//" in path),
        "has_hex_encoding": int(bool(re.search(r"%[0-9a-f]{2}", low))),
        "longest_token": max((len(t) for t in tokens), default=0),
        "num_tokens": len(tokens),
        # --- host / domain ---
        "has_https": int(p.scheme == "https"),
        "has_ip": int(bool(IP_RE.match(host))),
        "non_standard_port": int(port not in (None, 80, 443)),
        "is_shortener": int(host.replace("www.", "") in SHORTENERS),
        "risky_tld": int(tld in RISKY_TLDS),
        "tld_length": len(tld),
        "host_hyphens": host.count("-"),
        "host_digit_ratio": sum(ch.isdigit() for ch in host) / max(len(host), 1),
        "domain_vowel_ratio": vowels / max(len(main_domain), 1),
        # --- content / brand tricks ---
        "suspicious_words": sum(w in low for w in SUSPICIOUS),
        "brand_in_subdomain": int(any(b in subdomain for b in BRANDS)),
        "brand_in_path": int(any(b in path for b in BRANDS)),
        "url_entropy": entropy(url),
        "host_entropy": entropy(host),
        # --- additional URL risk signals ---
        "url_entropy_normalized": entropy(url) / max(math.log2(len(set(url))) if len(set(url)) > 1 else 1, 1),
        "has_fragment": int(bool(p.fragment)),
        "fragment_length": len(p.fragment),
        "has_credentials": int("@" in p.netloc),
        "has_punycode": int("xn--" in host),
        "has_unicode": int(any(ord(ch) > 127 for ch in url)),
        "has_https_token_in_host": int("https" in host),
        "host_is_local_or_reserved": int(host in {"localhost", "127.0.0.1"} or host.endswith(".local")),
        "domain_label_length": len(domain_label),
        "longest_subdomain_length": max((len(s) for s in parts[:-2]), default=0),
        "subdomain_entropy": entropy(subdomain),
        "path_segment_count": len(path_segments),
        "max_path_segment_length": max((len(s) for s in path_segments), default=0),
        "path_has_executable_extension": int(bool(re.search(r"\.(exe|scr|bat|cmd|msi|apk|jar)(?:$|\?)", path))),
        "path_has_archive_extension": int(bool(re.search(r"\.(zip|rar|7z|iso)(?:$|\?)", path))),
        "query_has_redirect_key": int(any(k in {"url", "redirect", "redirect_url", "next", "return", "returnurl", "continue", "dest", "destination", "target"} for k in query_keys)),
        "query_has_token_key": int(any(k in {"token", "session", "auth", "key", "password", "passwd", "otp"} for k in query_keys)),
        "query_key_count": len(query_keys),
        "query_value_length": sum(len(v) for _, v in query_pairs),
        "query_special_ratio": sum(not ch.isalnum() for ch in query) / max(len(query), 1),
        "has_suspicious_file_extension": int(bool(re.search(r"\.(php|asp|aspx|cgi)(?:$|\?)", path))),
        "brand_in_hostname": int(any(b in hostname_without_www for b in BRANDS)),
        "brand_domain_mismatch": brand_domain_mismatch,
        "contains_multiple_at_symbols": int(url.count("@") > 1),
        "contains_backslash": int("\\\\" in url),
        "contains_null_byte_encoding": int("%00" in low),
        "has_double_encoding": int(bool(re.search(r"%25[0-9a-f]{2}", low))),
        "suspicious_word_in_hostname": int(any(w in host for w in SUSPICIOUS)),
    }

X = pd.DataFrame([extract_features(u) for u in df["url"]])
y = df["label"].values
print(f"{X.shape[1]} features for {X.shape[0]} URLs")
X.head()

### Feature correlation
Highly correlated features carry similar information — useful to know when pruning.

In [ ]:
plt.figure(figsize=(13, 10))
sns.heatmap(X.corr(), cmap="coolwarm", center=0, square=True, cbar_kws={"shrink": .7})
plt.title("Feature correlation heatmap"); plt.tight_layout(); plt.show()

In [ ]:
# How do features differ between safe and malicious URLs?
plot_cols = ["url_length", "num_hyphens", "num_digits", "suspicious_words", "url_entropy", "host_digit_ratio"]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), plot_cols):
    sns.boxplot(x=y, y=X[col], ax=ax, palette=["#4c9", "#e55"])
    ax.set_xticklabels(["Safe", "Malicious"]); ax.set_xlabel(""); ax.set_title(col)
plt.tight_layout(); plt.show()

## 4. Train & compare models

In [ ]:
X_train, X_test, y_train, y_test, url_train, url_test = train_test_split(
    X, y, df["url"].values, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows, fitted, probas = [], {}, {}

def add_result(name, y_pred, proba, cv_f1=np.nan):
    probas[name] = proba
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, proba),
        "CV F1 (5-fold)": cv_f1,
    })

models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}
if HAS_XGB:
    models["XGBoost"] = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                                      eval_metric="logloss", random_state=RANDOM_STATE)

for name, model in models.items():
    cv_f1 = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1").mean()
    model.fit(X_train, y_train)
    fitted[name] = model
    add_result(name, model.predict(X_test), model.predict_proba(X_test)[:, 1], cv_f1)
print("Baseline models trained:", list(fitted))

### 4a. Hyperparameter tuning (Random Forest)
Instead of guessing settings, we let `RandomizedSearchCV` try many combinations.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "n_estimators": [100, 200, 400],
    "max_depth": [None, 8, 12, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", 0.5],
}
search = RandomizedSearchCV(RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
                            param_dist, n_iter=15, scoring="f1", cv=cv,
                            random_state=RANDOM_STATE, n_jobs=-1)
search.fit(X_train, y_train)
print("Best params:", search.best_params_)

tuned = search.best_estimator_
fitted["Random Forest (tuned)"] = tuned
add_result("Random Forest (tuned)", tuned.predict(X_test), tuned.predict_proba(X_test)[:, 1], search.best_score_)

### 4b. Text-based model (character n-grams)
Instead of hand-made features, this model reads the raw URL text — patterns like `-login`, `.xyz/` are learned automatically.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

text_model = make_pipeline(
    TfidfVectorizer(analyzer="char", ngram_range=(2, 4), min_df=2, max_features=50000),
    LogisticRegression(max_iter=1000))
cv_f1 = cross_val_score(text_model, url_train, y_train, cv=cv, scoring="f1").mean()
text_model.fit(url_train, y_train)
add_result("TF-IDF char n-grams + LogReg", text_model.predict(url_test),
           text_model.predict_proba(url_test)[:, 1], cv_f1)

In [ ]:
results = pd.DataFrame(rows).set_index("Model").round(4).sort_values("F1", ascending=False)
display(results)

# best model among those that use our hand-made features (used in the next sections)
best_name = [n for n in results.index if n in fitted][0]
print("\nBest feature-based model by F1:", best_name)

**How to read this:**
- **Precision** — of the URLs flagged malicious, how many really were? (low = many false alarms)
- **Recall** — of all malicious URLs, how many did we catch? (low = dangerous URLs slip through)
- For a security tool, **recall matters a lot** — missing a phishing URL is worse than a false alarm.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for name, proba in probas.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax[0].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, proba):.3f})")
ax[0].plot([0, 1], [0, 1], "k--", alpha=0.5)
ax[0].set_xlabel("False Positive Rate"); ax[0].set_ylabel("True Positive Rate")
ax[0].set_title("ROC curves"); ax[0].legend(fontsize=8)

ConfusionMatrixDisplay.from_estimator(fitted[best_name], X_test, y_test,
                                      display_labels=["Safe", "Malicious"], cmap="Blues", ax=ax[1])
ax[1].set_title(f"Confusion matrix — {best_name}")
plt.tight_layout(); plt.show()

### 4c. Learning curve
Does the model need more data, or is it already saturated / overfitting?

In [ ]:
from sklearn.model_selection import learning_curve

sizes, tr, va = learning_curve(fitted[best_name], X_train, y_train, cv=cv, scoring="f1",
                               train_sizes=np.linspace(0.1, 1.0, 6), n_jobs=-1)
plt.figure(figsize=(7, 4))
plt.plot(sizes, tr.mean(1), "o-", label="Training F1")
plt.plot(sizes, va.mean(1), "o-", label="Validation F1")
plt.xlabel("Training samples"); plt.ylabel("F1"); plt.title(f"Learning curve — {best_name}")
plt.legend(); plt.tight_layout(); plt.show()

### 4d. Threshold tuning
The default cut-off is 0.5, but for a security tool we may prefer **catching more threats** (higher recall) even at the cost of a few false alarms.

In [ ]:
from sklearn.metrics import precision_recall_curve

TARGET_RECALL = 0.98
prec, rec, thr = precision_recall_curve(y_test, probas[best_name])
ok = np.where(rec[:-1] >= TARGET_RECALL)[0]
BEST_THRESHOLD = float(thr[ok[-1]]) if len(ok) else 0.5   # highest threshold that still meets the recall target
print(f"Chosen threshold: {BEST_THRESHOLD:.3f}  (target recall >= {TARGET_RECALL})")

plt.figure(figsize=(7, 4))
plt.plot(rec, prec); plt.scatter(rec[ok[-1]] if len(ok) else 0, prec[ok[-1]] if len(ok) else 0, c="red", zorder=3, label="chosen")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Precision-Recall curve"); plt.legend()
plt.tight_layout(); plt.show()

pred_thr = (probas[best_name] >= BEST_THRESHOLD).astype(int)
print(confusion_matrix(y_test, pred_thr), "\n(rows = actual, cols = predicted)")
print(f"Precision {precision_score(y_test, pred_thr):.3f} | Recall {recall_score(y_test, pred_thr):.3f}")

## 5. Error analysis — where does the best model go wrong?

In [ ]:
best_pred = fitted[best_name].predict(X_test)
err = pd.DataFrame({"url": url_test, "actual": y_test, "predicted": best_pred})
err["type"] = np.where((err.actual == 1) & (err.predicted == 0), "False Negative (missed threat)",
              np.where((err.actual == 0) & (err.predicted == 1), "False Positive (false alarm)", "Correct"))
print(err["type"].value_counts(), "\n")
pd.set_option("display.max_colwidth", 90)
err[err["type"] != "Correct"].head(15)

## 5b. Security-focused evaluation

Accuracy can hide missed attacks, so this section adds **specificity, balanced accuracy, Matthews correlation coefficient (MCC), and a full classification report**. These metrics are still only as trustworthy as the labels and dataset; synthetic demo data is not suitable for security claims.


In [ ]:
from sklearn.metrics import classification_report, balanced_accuracy_score, matthews_corrcoef

evaluation_rows = []
for name, model in fitted.items():
    pred = model.predict(X_test)
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    evaluation_rows.append({
        "Model": name,
        "Specificity": tn / max(tn + fp, 1),
        "Balanced accuracy": balanced_accuracy_score(y_test, pred),
        "MCC": matthews_corrcoef(y_test, pred),
        "False negatives": int(fn),
        "False positives": int(fp),
    })
security_metrics = pd.DataFrame(evaluation_rows).set_index("Model").sort_values("MCC", ascending=False)
display(security_metrics.round(4))
print("Classification report for best feature-based model:")
print(classification_report(y_test, fitted[best_name].predict(X_test),
                            target_names=["Safe", "Malicious"], zero_division=0))


## 5c. Probability calibration

A score of 0.8 should not automatically be interpreted as an 80% real-world probability. The calibration plot below checks whether predicted scores align with observed frequencies on this test split. Small or synthetic datasets can make this plot misleading.


In [ ]:
from sklearn.calibration import calibration_curve

cal_true, cal_pred = calibration_curve(y_test, probas[best_name], n_bins=10, strategy="quantile")
plt.figure(figsize=(6, 5))
plt.plot(cal_pred, cal_true, marker="o", label=best_name)
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
plt.xlabel("Mean predicted score")
plt.ylabel("Observed malicious fraction")
plt.title("Probability calibration")
plt.legend()
plt.tight_layout()
plt.show()


## 7b. Explainable single-URL scanner

This wrapper returns the verdict, risk band, score, and a compact set of URL indicators for manual review. Indicators are heuristic clues—not proof that a URL is malicious or safe. Avoid opening suspicious URLs directly.


In [ ]:
def scan_detailed(url, threshold=None):
    threshold = BEST_THRESHOLD if threshold is None else float(threshold)
    features = extract_features(str(url))
    frame = pd.DataFrame([features]).reindex(columns=X.columns, fill_value=0)
    probability = float(fitted[best_name].predict_proba(frame)[0, 1])
    risk_band = "HIGH" if probability >= max(threshold, 0.75) else (
        "MEDIUM" if probability >= min(threshold, 0.5) else "LOW")
    indicators = []
    indicator_map = {
        "has_ip": "Uses an IP address as host",
        "is_shortener": "Uses a known URL shortener",
        "risky_tld": "Uses a TLD in the configured watchlist",
        "has_credentials": "Contains an @ symbol in the authority section",
        "has_punycode": "Contains punycode in the hostname",
        "brand_domain_mismatch": "Brand-like text appears in a non-matching hostname",
        "suspicious_word_in_hostname": "Contains a watchlisted word in the hostname",
        "query_has_redirect_key": "Contains a redirect-style query parameter",
        "has_https_token_in_host": "Hostname contains the text 'https'",
        "has_double_encoding": "Contains a double-encoding pattern",
    }
    for key, description in indicator_map.items():
        if features.get(key, 0):
            indicators.append(description)
    return {
        "url": str(url),
        "malicious_score": round(probability, 4),
        "verdict": "MALICIOUS-LIKELY" if probability >= threshold else "SAFE-LIKELY",
        "risk_band": risk_band,
        "threshold": round(threshold, 4),
        "indicators": indicators or ["No configured heuristic indicators triggered"],
        "note": "ML estimate only; verify using trusted security tools and current threat intelligence."
    }

display(pd.DataFrame([
    scan_detailed("https://www.python.org/docs"),
    scan_detailed("http://paypal.secure-login-verify.xyz/update?id=991")
]))


## 7c. Batch scan summary and risk triage

Adds risk bands and a short summary to the existing CSV batch scanner. Results should be reviewed by a person before taking action.


In [ ]:
def scan_csv_with_summary(path_in, path_out="scan_results_detailed.csv", threshold=None):
    data = pd.read_csv(path_in)
    if "url" not in data.columns:
        raise ValueError("Input CSV must contain a 'url' column.")
    details = [scan_detailed(str(u), threshold=threshold) for u in data["url"].fillna("")]
    detail_df = pd.DataFrame(details).drop(columns=["indicators", "note"])
    result = data.copy()
    result["malicious_score"] = detail_df["malicious_score"].values
    result["verdict"] = detail_df["verdict"].values
    result["risk_band"] = detail_df["risk_band"].values
    result["review_indicators"] = ["; ".join(d["indicators"]) for d in details]
    result.to_csv(path_out, index=False)
    print(f"Scanned {len(result)} URLs -> {path_out}")
    display(result["risk_band"].value_counts().rename_axis("Risk band").to_frame("Count"))
    return result

# Example: scan_csv_with_summary("sample_urls.csv")


## 6. Feature importance

In [ ]:
model = fitted[best_name]
if hasattr(model, "feature_importances_"):
    imp = pd.Series(model.feature_importances_, index=X.columns)
else:  # Logistic Regression pipeline
    imp = pd.Series(np.abs(model[-1].coef_[0]), index=X.columns)

imp.sort_values().plot(kind="barh", figsize=(8, 6), color="#58a")
plt.title(f"Feature importance — {best_name}")
plt.xlabel("Importance"); plt.tight_layout(); plt.show()

### Permutation importance (more reliable)
Shuffle one feature at a time and see how much the score drops — works for any model.

In [ ]:
from sklearn.inspection import permutation_importance

pi = permutation_importance(fitted[best_name], X_test, y_test, scoring="f1",
                            n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
pi_s = pd.Series(pi.importances_mean, index=X.columns).sort_values().tail(15)
pi_s.plot(kind="barh", figsize=(8, 6), color="#c85")
plt.title("Top 15 features — permutation importance (F1 drop)")
plt.xlabel("Mean F1 drop"); plt.tight_layout(); plt.show()

## 7. Save the best model

In [ ]:
import joblib
joblib.dump({"model": fitted[best_name], "features": list(X.columns)}, "cyberguard_best_model.joblib")

def scan(url, threshold=None):
    threshold = BEST_THRESHOLD if threshold is None else threshold
    feats = pd.DataFrame([extract_features(url)])[list(X.columns)]
    p = fitted[best_name].predict_proba(feats)[0, 1]
    return {"url": url, "malicious_probability": round(float(p), 3),
            "verdict": "⚠️ MALICIOUS" if p >= threshold else "✅ SAFE"}

for u in ["https://www.python.org/docs", "http://192.168.4.77/secure-login/verify?id=123&ref=@9"]:
    print(scan(u))

### Batch scanning
Scan a whole CSV of URLs (column `url`) and save the verdicts.

In [ ]:
def scan_csv(path_in, path_out="scan_results.csv"):
    data = pd.read_csv(path_in)
    feats = pd.DataFrame([extract_features(str(u)) for u in data["url"]])[list(X.columns)]
    data["malicious_probability"] = fitted[best_name].predict_proba(feats)[:, 1].round(3)
    data["verdict"] = np.where(data["malicious_probability"] >= BEST_THRESHOLD, "MALICIOUS", "SAFE")
    data.to_csv(path_out, index=False)
    return data

pd.DataFrame({"url": ["https://github.com/", "http://paypal.secure-login-verify.xyz/update?id=991"]}).to_csv("sample_urls.csv", index=False)
scan_csv("sample_urls.csv")

## 9. Conclusion and responsible use

- Compared baseline, tuned, and character n-gram models using accuracy, precision, recall, F1 and ROC-AUC.
- Expanded URL features with hostname/brand mismatch clues, punycode, redirect parameters, file extensions, fragments, encoding patterns, and path/query statistics.
- Added security-focused metrics (specificity, balanced accuracy, MCC), classification reporting, probability calibration, detailed single-URL triage, and a batch summary.
- Threshold tuning supports a recall/false-alarm trade-off; select thresholds using validation data rather than repeatedly tuning on the final test set.
- **Important:** this is a research/demo classifier, not a replacement for a browser warning system or threat-intelligence service. Demo data produces illustrative results only. Do not treat a low score as proof of safety.

**Next steps:** evaluate on a time-separated, deduplicated real-world dataset; combine handcrafted and character n-gram features in a single pipeline; consider domain-age/DNS/reputation signals only from trusted services; and add a Streamlit or CLI interface.

---
*Part of the CyberGuard ML URL Scanner project — Harpreet Singh*


## 10. Live URL reputation checks (optional)

This section adds **read-only reputation lookups** against Google Safe Browsing (non-commercial use) and VirusTotal URL reports. It does not visit the submitted URL or automatically submit new URLs for scanning.

**Privacy:** reputation providers receive the URL being checked (including its path/query). Do not look up confidential, private, or personal URLs. VirusTotal documents that queried indicators can be added to its shared dataset. Use only URLs you are allowed to disclose. The providers' terms, quotas, and eligibility apply.

**Key safety:** put credentials in environment variables or a local `.env` file excluded from version control. Never paste keys into a notebook cell or commit them.

In [ ]:
# Install once if needed: %pip install requests python-dotenv
import os, base64, ipaddress
from urllib.parse import urlsplit
import requests

# Optional: load a local .env file. Environment variables already set by your host take precedence.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

VT_API_KEY = os.getenv("VIRUSTOTAL_API_KEY")
GOOGLE_SAFE_BROWSING_API_KEY = os.getenv("GOOGLE_SAFE_BROWSING_API_KEY")
REPUTATION_TIMEOUT_SECONDS = 12


def _validate_public_url(url):
    """Validate syntax only; never fetch or follow the submitted URL."""
    if not isinstance(url, str) or not url.strip() or len(url) > 2048:
        raise ValueError("URL must be a non-empty string of at most 2048 characters.")
    candidate = url.strip()
    parts = urlsplit(candidate)
    if parts.scheme.lower() not in {"http", "https"} or not parts.hostname:
        raise ValueError("Only absolute http:// and https:// URLs are accepted.")
    if parts.username or parts.password:
        raise ValueError("URLs containing embedded credentials are not accepted.")
    try:
        parts.port  # trigger validation for malformed ports
    except ValueError as exc:
        raise ValueError("URL contains an invalid port.") from exc
    return candidate


def check_google_safe_browsing(url, api_key=None):
    """Check known threat lists. Google receives the submitted URL."""
    key = api_key or os.getenv("GOOGLE_SAFE_BROWSING_API_KEY")
    if not key:
        return {"provider": "Google Safe Browsing", "status": "not_configured", "matches": [],
                "message": "Set GOOGLE_SAFE_BROWSING_API_KEY to enable this provider."}
    candidate = _validate_public_url(url)
    endpoint = "https://safebrowsing.googleapis.com/v4/threatMatches:find"
    payload = {
        "client": {"clientId": "cyberguard-url-scanner", "clientVersion": "1.0"},
        "threatInfo": {
            "threatTypes": ["MALWARE", "SOCIAL_ENGINEERING", "UNWANTED_SOFTWARE", "POTENTIALLY_HARMFUL_APPLICATION"],
            "platformTypes": ["ANY_PLATFORM"],
            "threatEntryTypes": ["URL"],
            "threatEntries": [{"url": candidate}],
        },
    }
    try:
        response = requests.post(endpoint, params={"key": key}, json=payload,
                                 timeout=REPUTATION_TIMEOUT_SECONDS)
        if response.status_code == 429:
            return {"provider": "Google Safe Browsing", "status": "rate_limited", "matches": [],
                    "message": "Provider quota/rate limit reached; retry later."}
        response.raise_for_status()
        data = response.json()
        matches = data.get("matches", [])
        return {"provider": "Google Safe Browsing", "status": "malicious" if matches else "no_match",
                "matches": [{"threatType": m.get("threatType"), "platformType": m.get("platformType")}
                            for m in matches],
                "message": "Known threat match found." if matches else "No known threat match returned; this does not prove the URL is safe."}
    except requests.Timeout:
        return {"provider": "Google Safe Browsing", "status": "error", "matches": [], "message": "Request timed out."}
    except requests.RequestException as exc:
        status = getattr(getattr(exc, "response", None), "status_code", None)
        return {"provider": "Google Safe Browsing", "status": "error", "matches": [],
                "message": f"Provider request failed (HTTP {status}). Check configuration and quota."}
    except ValueError:
        return {"provider": "Google Safe Browsing", "status": "error", "matches": [], "message": "Provider returned an invalid response."}


def check_virustotal_report(url, api_key=None):
    """Read an existing VirusTotal URL report only; does not submit a new scan."""
    key = api_key or os.getenv("VIRUSTOTAL_API_KEY")
    if not key:
        return {"provider": "VirusTotal", "status": "not_configured", "message": "Set VIRUSTOTAL_API_KEY to enable this provider."}
    candidate = _validate_public_url(url)
    url_id = base64.urlsafe_b64encode(candidate.encode("utf-8")).decode("ascii").rstrip("=")
    try:
        response = requests.get(f"https://www.virustotal.com/api/v3/urls/{url_id}",
                                headers={"x-apikey": key, "accept": "application/json"},
                                timeout=REPUTATION_TIMEOUT_SECONDS)
        if response.status_code == 404:
            return {"provider": "VirusTotal", "status": "not_found", "message": "No existing report found; this integration does not submit new scans."}
        if response.status_code == 429:
            return {"provider": "VirusTotal", "status": "rate_limited", "message": "Provider quota/rate limit reached; retry later."}
        response.raise_for_status()
        attrs = response.json().get("data", {}).get("attributes", {})
        stats = attrs.get("last_analysis_stats", {})
        malicious = int(stats.get("malicious", 0) or 0)
        suspicious = int(stats.get("suspicious", 0) or 0)
        status = "malicious" if malicious else ("suspicious" if suspicious else "no_detection")
        return {"provider": "VirusTotal", "status": status,
                "last_analysis_stats": stats,
                "reputation": attrs.get("reputation"),
                "last_analysis_date": attrs.get("last_analysis_date"),
                "message": "Existing community report; absence of detections does not prove safety."}
    except requests.Timeout:
        return {"provider": "VirusTotal", "status": "error", "message": "Request timed out."}
    except requests.RequestException as exc:
        status = getattr(getattr(exc, "response", None), "status_code", None)
        return {"provider": "VirusTotal", "status": "error", "message": f"Provider request failed (HTTP {status}). Check configuration and quota."}
    except (ValueError, TypeError):
        return {"provider": "VirusTotal", "status": "error", "message": "Provider returned an invalid response."}


def check_live_reputation(url, providers=("google", "virustotal")):
    """Combine selected reputation checks. Errors remain errors, never 'safe'."""
    candidate = _validate_public_url(url)
    results = []
    if "google" in providers:
        results.append(check_google_safe_browsing(candidate))
    if "virustotal" in providers:
        results.append(check_virustotal_report(candidate))
    known_bad = any(r.get("status") in {"malicious", "suspicious"} for r in results)
    configured = [r for r in results if r.get("status") not in {"not_configured", "error", "rate_limited"}]
    if known_bad:
        verdict = "THREAT INDICATED"
    elif configured and all(r.get("status") in {"no_match", "no_detection", "not_found"} for r in configured):
        verdict = "NO KNOWN THREAT FOUND — NOT A SAFETY GUARANTEE"
    else:
        verdict = "INCONCLUSIVE — CHECK PROVIDER CONFIGURATION/STATUS"
    return {"url": candidate, "verdict": verdict, "provider_results": results,
            "note": "Reputation checks complement, not replace, the ML score and human review."}

print("Live reputation helpers ready. Configured providers:",
      [name for name, value in [("Google Safe Browsing", GOOGLE_SAFE_BROWSING_API_KEY),
                                ("VirusTotal", VT_API_KEY)] if value])

### Example: run reputation checks safely

This example uses a public demonstration URL. The code does not open the target website. Provider requests only occur when you execute the cell and have configured the relevant key. Do not use sensitive URLs because providers receive the URL being queried.

In [ ]:
# Example (uncomment to make live API requests after configuring your keys):
# reputation = check_live_reputation("https://www.python.org/")
# display(reputation)

# Optional: combine the existing ML scan with live reputation, without hiding provider errors.
def scan_with_reputation(url, threshold=None, providers=("google", "virustotal")):
    ml_result = scan_detailed(url, threshold=threshold)
    reputation_result = check_live_reputation(url, providers=providers)
    return {"ml_result": ml_result, "reputation_result": reputation_result}

# Example:
# display(scan_with_reputation("https://www.python.org/"))

### API key setup

1. Create API credentials directly with the provider: [VirusTotal API docs](https://docs.virustotal.com/docs/api-overview) and [Google Safe Browsing API docs](https://developers.google.com/safe-browsing/v4/lookup-api).
2. Copy `.env.example` to `.env` and fill the values locally, or set environment variables in your deployment platform's secret manager.
3. Add `.env` to `.gitignore`; never commit credentials or print them in notebook outputs.
4. Google Safe Browsing v4 is intended for non-commercial use; commercial products should evaluate Google Web Risk. Respect provider privacy terms and rate limits.
5. VirusTotal public API has strict quotas and terms. This integration only reads existing reports and deliberately does not submit new URL scans.